# Reviewer: orientation and direction tuning of depth-tuned neurons

> How does depth sensitivity relate to other sensitivities, such as orientation? Are these multiplexed,
> or are they separate sub-populations?

Six figure sessions (mice PZAH10.2d and PZAH10.2f) have drifting-grating recordings (`SFTF`) after the
depth protocol. The gratings ran as separate recordings, so no stationary-epoch selection is needed:
all grating trials are used. ROIs are the same suite2p ROIs as the depth recordings.

- **Stimulus**: full-field drifting gratings, 6 SF (0.01-0.32 cpd) × 6 TF (0.5-16 Hz) × 8 directions,
  ~4 s each with no blank, each recording showing the full grid once (5 recordings per session).
- **Response**: mean dF/F (`ast_neuropil=False` traces, as in the manuscript) over the trial, shifted by
  `LAG` imaging volumes.
- **Tuning**: for each ROI, the preferred SF/TF is the combination with the largest direction-averaged
  response; the direction tuning curve is taken at that SF/TF. Preferred direction and orientation, gDSI
  and gOSI come from the vector sum of the rectified curve. Orientation is the direction of motion mod 180°.
- **Significance**: *responsive* = one-way ANOVA across all 288 conditions; *orientation / direction
  selective* = responsive and gOSI / gDSI above a shuffle of direction labels among the trials of the
  preferred SF/TF. Shuffling within an SF/TF leaves its direction-averaged response unchanged, so the
  choice of preferred SF/TF does not bias the test.
- **Parametric fit** (summary figure): `neurons_df_sftf.pickle` of each session, from the pipeline
  (`depth_SFTF`, rerun in October 2026 with `ast_neuropil=False` dF/F and leave-one-recording-out
  cross-validation). *SF/TF tuned* = one-sided Spearman p < 0.05 between held-out responses and
  predictions, the same test as for depth tuning.
- **Depth tuning**: `depth_tuned` from `figure_utils.rf_distance.load_session_neurons` (one-sided Spearman
  on the closed-loop dF/F depth tuning, as in the figures). Only `iscell` ROIs are kept.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import flexiznam as flz

from cottage_analysis.preprocessing import synchronisation
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM
from v1_depth_map.figure_utils.rf_distance import load_session_neurons
from v1_depth_map.paths import get_figures_roots

style.setup_figure_fonts()

PROJECT = "hey2_3d-vision_foodres_20220101"
SESSIONS = [
    "PZAH10.2d_S20230920",
    "PZAH10.2d_S20230922",
    "PZAH10.2f_S20230817",
    "PZAH10.2f_S20230912",
    "PZAH10.2f_S20230914",
    "PZAH10.2f_S20230924",
]
PHOTODIODE_PROTOCOL = 5  # all PZAH mice
TRACES = {"anatomical_only": 3, "ast_neuropil": False}
PARAMS = ["SpatialFrequency", "TemporalFrequency", "Angle"]
LAG = 2  # imaging volumes between stimulus and response window (as in gratings.generate_trials_df)
MIN_VOLUMES = 10  # drop trials shorter than this (truncated at the end of a recording)
P_RESPONSIVE = 0.01  # ANOVA across conditions
P_SELECTIVE = 0.05  # gOSI / gDSI shuffle
N_SHUFFLES = 1000
REBUILD = False  # True: re-sync the recordings instead of reading the trial cache
# selection criteria and stats of the thesis SFTF figure (yiran-phd-thesis/yh_thesis/depth/SFTF.ipynb),
# except SF/TF tuned, which is cross-validated (one-sided Spearman p < 0.05, as for depth)
RSOF_RSQ_THR = 0.02  # RS/OF tuned (panels B, C): rsof_test_rsq_closedloop_g2d above this
DSI_THR = 0.6  # panel F: direction-selective cells only
N_BOOTS = 20000  # hierarchical bootstrap (mouse, session) of the B, C correlations

GROUPS = {"depth tuned": "#2a78d6", "not depth tuned": "#eb6834"}

flm = flz.get_flexilims_session(PROJECT)
_, SAVE_ROOT = get_figures_roots(flm, fig_subdir="fig_reviewer_gratings")
CACHE = SAVE_ROOT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print(f"Saving to {SAVE_ROOT}")

## Grating trials

One row per trial, response = mean dF/F of each ROI over the trial window. Trials are split where the
parameter-log time (`stimulus_harptime`) changes, so two consecutive trials with the same parameters stay
separate. The last trial of a recording gets the median trial length. The trial tables are cached
per session.

In [ ]:
def recording_trials(recording):
    # Trial table and (n_trials, n_rois) mean dF/F of one SFTF recording.
    vs_df = synchronisation.generate_vs_df(
        recording=recording,
        photodiode_protocol=PHOTODIODE_PROTOCOL,
        flexilims_session=flm,
        project=PROJECT,
    )
    img_df = synchronisation.generate_imaging_df(
        vs_df=vs_df,
        recording=recording,
        flexilims_session=flm,
        filter_datasets=TRACES,
        return_volumes=True,
    ).reset_index(drop=True)
    dff = np.vstack(img_df.dffs.values)  # (n_volumes, n_rois)

    stim = img_df["stimulus_harptime"]
    onset = (stim.notna() & (stim != stim.shift())).values
    starts = np.flatnonzero(onset)
    ends = np.append(starts[1:], starts[-1] + int(np.median(np.diff(starts))))
    starts, ends = starts + LAG, np.minimum(ends + LAG, len(img_df))
    keep = ends - starts >= MIN_VOLUMES

    trials = img_df.loc[onset, PARAMS].reset_index(drop=True)
    trials["n_volumes"] = ends - starts
    trials["recording"] = recording["name"]
    resp = np.stack([dff[s:e].mean(axis=0) for s, e in zip(starts, ends)])
    return trials[keep].reset_index(drop=True), resp[keep]


def session_trials(session):
    # All SFTF trials of a session, cached in CACHE.
    path = CACHE / f"{session}_sftf_trials.pickle"
    if path.exists() and not REBUILD:
        return pd.read_pickle(path)
    recordings = flz.get_children(
        parent_name=session, children_datatype="recording", flexilims_session=flm
    )  # indexed by name, which the sync functions read as `recording.name`
    recordings = recordings[recordings["name"].str.contains("SFTF")].sort_index()
    out = [recording_trials(rec) for _, rec in recordings.iterrows()]
    data = dict(
        trials=pd.concat([o[0] for o in out], ignore_index=True),
        dff=np.concatenate([o[1] for o in out], axis=0),
    )
    pd.to_pickle(data, path)
    return data


data = {session: session_trials(session) for session in SESSIONS}

In [ ]:
summary = []
for session, d in data.items():
    t = d["trials"]
    reps = t.groupby(PARAMS).size()
    summary.append(
        dict(
            session=session,
            recordings=t.recording.nunique(),
            trials=len(t),
            rois=d["dff"].shape[1],
            conditions=len(reps),
            reps_min=reps.min(),
            reps_max=reps.max(),
            volumes_per_trial=t.n_volumes.median(),
        )
    )
summary = pd.DataFrame(summary)
summary

## Tuning metrics

In [ ]:
def vector_metrics(curves, theta):
    # gOSI, gDSI, preferred orientation and direction (deg) of rectified curves (..., n_dir).
    r = np.clip(curves, 0, None)
    total = r.sum(axis=-1)
    v1 = (r * np.exp(1j * theta)).sum(axis=-1)
    v2 = (r * np.exp(2j * theta)).sum(axis=-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        gosi, gdsi = np.abs(v2) / total, np.abs(v1) / total
    pref_ori = np.rad2deg(np.angle(v2)) / 2 % 180
    pref_dir = np.rad2deg(np.angle(v1)) % 360
    return gosi, gdsi, pref_ori, pref_dir


def session_tuning(trials, resp, rng):
    values = [np.unique(trials[p]) for p in PARAMS]
    n_sf, n_tf, n_dir = (len(v) for v in values)
    i_sf, i_tf, i_dir = (np.searchsorted(v, trials[p]) for v, p in zip(values, PARAMS))
    i_sftf = i_sf * n_tf + i_tf
    cond = i_sftf * n_dir + i_dir
    n_cond, n_roi = n_sf * n_tf * n_dir, resp.shape[1]
    theta = np.deg2rad(values[2])

    counts = np.bincount(cond, minlength=n_cond)
    sums = np.zeros((n_cond, n_roi))
    np.add.at(sums, cond, resp)
    with np.errstate(invalid="ignore"):
        means = (sums / counts[:, None]).reshape(n_sf * n_tf, n_dir, n_roi)
    best = np.nanargmax(np.nanmean(means, axis=1), axis=0)  # preferred SF/TF per ROI
    curves = means[best, :, np.arange(n_roi)]  # (n_roi, n_dir)
    gosi, gdsi, pref_ori, pref_dir = vector_metrics(curves, theta)

    # shuffle direction labels among the trials of each ROI's preferred SF/TF
    p_osi, p_dsi = np.full(n_roi, np.nan), np.full(n_roi, np.nan)
    for k in np.unique(best):
        rois = np.flatnonzero(best == k)
        tr = np.flatnonzero(i_sftf == k)
        labels = np.tile(i_dir[tr], (N_SHUFFLES, 1))
        onehot = np.eye(n_dir)[rng.permuted(labels, axis=1)]  # (n_shuffles, n_trials, n_dir)
        onehot /= np.bincount(i_dir[tr], minlength=n_dir)
        shuffled = np.einsum("std,tr->srd", onehot, resp[np.ix_(tr, rois)])
        osi_sh, dsi_sh, _, _ = vector_metrics(shuffled, theta)
        p_osi[rois] = (1 + (osi_sh >= gosi[rois]).sum(axis=0)) / (N_SHUFFLES + 1)
        p_dsi[rois] = (1 + (dsi_sh >= gdsi[rois]).sum(axis=0)) / (N_SHUFFLES + 1)

    p_resp = stats.f_oneway(*[resp[cond == c] for c in range(n_cond)], axis=0).pvalue
    return pd.DataFrame(
        dict(
            roi=np.arange(n_roi),
            pref_sf=values[0][best // n_tf],
            pref_tf=values[1][best % n_tf],
            gosi=gosi,
            gdsi=gdsi,
            pref_ori=pref_ori,
            pref_dir=pref_dir,
            p_osi=p_osi,
            p_dsi=p_dsi,
            p_resp=p_resp,
        )
    ), curves


rng = np.random.default_rng(0)
cells, curves = [], {}
for session in SESSIONS:
    neurons_df, _ = load_session_neurons(flm, session)
    tuning, curves[session] = session_tuning(
        data[session]["trials"], data[session]["dff"], rng
    )
    assert len(tuning) == len(neurons_df), f"{session}: ROI count differs from neurons_df"
    tuning["session"] = session
    tuning["iscell"] = neurons_df.iscell.values.astype(bool)
    tuning["depth_tuned"] = neurons_df.depth_tuned.values.astype(bool)
    tuning["preferred_depth"] = neurons_df.preferred_depth_closedloop.values
    # RS/OF Gaussian fit of the depth protocol, rad/s -> deg/s
    tuning["preferred_of"] = np.degrees(neurons_df.preferred_OF_closedloop_g2d.astype(float).values)
    tuning["rsof_test_rsq"] = neurons_df.rsof_test_rsq_closedloop_g2d.astype(float).values
    cells.append(tuning)
cells = pd.concat(cells, ignore_index=True)
cells = cells[cells.iscell].reset_index(drop=True)
cells["group"] = np.where(cells.depth_tuned, "depth tuned", "not depth tuned")
cells["responsive"] = cells.p_resp < P_RESPONSIVE
cells["ori_selective"] = cells.responsive & (cells.p_osi < P_SELECTIVE)
cells["dir_selective"] = cells.responsive & (cells.p_dsi < P_SELECTIVE)
print(f"{len(cells)} cells")

## Parametric fit

`neurons_df_sftf.pickle` of each session (see `revisions/sftf/NEURONS_DF_SFTF.md`): `fit_sftf_tuning` on
single-trial responses, a 2D Gaussian in log SF × log TF times a von Mises direction tuning with a direction
selectivity parameter `dsi` (0: equal response to both directions of an orientation, 1: no response to the
opposite direction). Preferred speed = TF0 / SF0 (deg/s), preferred direction = `alpha0`, preferred
orientation = `alpha0` mod 180°. Each ROI is also fitted on 4 of the 5 recordings and evaluated on the
5th; a cell is *SF/TF tuned* if the Spearman correlation between held-out responses and predictions is
significantly positive (one-sided p < 0.05, `add_one_sided_spearman_significance`, as for depth tuning).

In [ ]:
from cottage_analysis.analysis import common_utils
from cottage_analysis.pipelines import pipeline_utils


def load_sftf_fit(session):
    folder = pipeline_utils.create_neurons_ds(
        session_name=session, flexilims_session=flm, project=PROJECT, conflicts="skip"
    ).path_full.parent
    fit = pd.read_pickle(folder / "neurons_df_sftf.pickle")
    assert "sftf_test_spearmanr_pval" in fit, f"{session}: neurons_df_sftf has no cross-validation"
    fit.insert(0, "roi", np.arange(len(fit)))
    fit.insert(0, "session", session)
    return fit.drop(columns=["sftf_test_popts"])


fits = pd.concat([load_sftf_fit(s) for s in SESSIONS], ignore_index=True)
cells = cells.drop(columns=[c for c in cells.columns if c.startswith("fit_")]).merge(
    fits.rename(columns={c: f"fit_{c}" for c in fits.columns if c not in ("session", "roi")}),
    on=["session", "roi"],
    how="left",
    validate="one_to_one",
)
cells["pref_speed"] = np.exp(cells.fit_tf0 - cells.fit_sf0)  # deg/s
cells["fit_dir"] = np.rad2deg(cells.fit_alpha0) % 360
cells["fit_ori"] = cells.fit_dir % 180
common_utils.add_one_sided_spearman_significance(
    cells,
    rval_col="fit_sftf_test_spearmanr_rval",
    pval_col="fit_sftf_test_spearmanr_pval",
    out_col="sftf_tuned",
)
cells["rsof_tuned"] = cells.rsof_test_rsq > RSOF_RSQ_THR
cells.to_csv(SAVE_ROOT / "grating_tuning_cells.csv", index=False)
print(f"SF/TF tuned (cross-validated): {cells.sftf_tuned.sum()} / {len(cells)}")
print(f"  for reference, in-sample R² > 0.05 (thesis): {(cells.fit_rsq > 0.05).sum()}")
print(f"RS/OF tuned (test R² > {RSOF_RSQ_THR}): {cells.rsof_tuned.sum()}")
print(f"SF/TF tuned and depth selective: {(cells.sftf_tuned & cells.depth_tuned).sum()}")
print(f"SF/TF tuned and RS/OF tuned: {(cells.sftf_tuned & cells.rsof_tuned).sum()}")

In [ ]:
# in-sample vs held-out fit quality
fig, axes = plt.subplots(1, 2, figsize=(12 * CM, 5 * CM))
ax = axes[0]
ax.scatter(cells.fit_rsq, cells.fit_sftf_test_rsq, s=3, color="k", alpha=0.3, linewidths=0)
lim = [min(cells.fit_sftf_test_rsq.min(), 0), cells.fit_rsq.max()]
ax.plot(lim, lim, color="#8a8984", lw=0.8, ls="--")
ax.set_xlabel("In-sample R²")
ax.set_ylabel("Held-out R²")
ax = axes[1]
ax.hist(cells.fit_sftf_test_spearmanr_rval, bins=50, color="k")
ax.set_xlabel("Held-out Spearman r")
ax.set_ylabel("Number of cells")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
style.savefig(SAVE_ROOT / "sftf_crossval.pdf", fig=fig, bbox_inches="tight")

## Summary figure

- **A**: overlap of depth-selective and SF/TF-tuned cells (areas proportional to cell counts).
- **B, C**: preferred grating speed vs preferred optic flow speed and preferred virtual depth (depth
  protocol), for cells that are SF/TF tuned and RS/OF tuned (thesis selection; fits on a bound are kept).
  Spearman ρ with a hierarchical bootstrap over mice and sessions (`N_BOOTS`), as in the thesis.
  Dashed line in B: identity.
- **D, E**: direction selectivity (`dsi`) and preferred orientation of SF/TF-tuned, depth-selective cells.
- **F**: preferred direction of SF/TF-tuned, depth-selective cells with `dsi` > `DSI_THR`.

In [ ]:
from scipy.optimize import brentq


def circle_overlap(r1, r2, d):
    # Area of the intersection of two circles of radii r1, r2 with centres d apart.
    if d >= r1 + r2:
        return 0.0
    if d <= abs(r1 - r2):
        return np.pi * min(r1, r2) ** 2
    a1 = r1**2 * np.arccos((d**2 + r1**2 - r2**2) / (2 * d * r1))
    a2 = r2**2 * np.arccos((d**2 + r2**2 - r1**2) / (2 * d * r2))
    return a1 + a2 - 0.5 * np.sqrt((-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2))


def plot_venn(ax, n_all, n_a, n_b, n_ab, names, colors):
    # Area-proportional Venn of sets a and b inside the set of all cells.
    R, ra, rb = np.sqrt(np.array([n_all, n_a, n_b]) / np.pi)
    lo, hi = abs(ra - rb), ra + rb
    d = brentq(lambda x: circle_overlap(ra, rb, x) - n_ab, lo + 1e-9, hi - 1e-9)
    xa = -(ra + d + rb) / 2 + ra
    xb = xa + d
    # highest position keeping both circles inside the outer one
    y = 0.97 * min(np.sqrt(max((R - r) ** 2 - x**2, 0)) for r, x in [(ra, xa), (rb, xb)])
    ax.add_patch(plt.Circle((0, 0), R, color="#d9d8d4", lw=0))
    ax.add_patch(plt.Circle((xa, y), ra, color=colors[0], alpha=0.6, lw=0))
    ax.add_patch(plt.Circle((xb, y), rb, color=colors[1], alpha=0.6, lw=0))
    pct = lambda n: f"{100 * n / n_all:.0f}%"
    for x, n in [
        ((xa - ra + xb - rb) / 2, n_a - n_ab),
        ((xb - rb + xa + ra) / 2, n_ab),
        ((xa + ra + xb + rb) / 2, n_b - n_ab),
    ]:
        ax.text(x, y, pct(n), ha="center", va="center")
    ax.text(0, -0.75 * R, pct(n_all - n_a - n_b + n_ab), ha="center", va="center")
    ax.text(xa - ra, y + ra, names[0], ha="right", va="bottom")
    ax.text(xb + 0.3 * rb, y + rb, names[1], ha="left", va="bottom")
    ax.text(0, -R * 1.02, f"All neurons (n={n_all})", ha="center", va="top")
    ax.set_xlim(-R * 1.05, R * 1.05)
    ax.set_ylim(-R * 1.25, R * 1.15)
    ax.set_aspect("equal")
    ax.axis("off")


def polar_hist(ax, values_deg, period, n_bins):
    # Bars centred on multiples of period / n_bins (the stimulus directions when n_bins = 8).
    width = period / n_bins
    counts, _ = np.histogram((values_deg + width / 2) % period, bins=n_bins, range=(0, period))
    centres = np.deg2rad(np.arange(n_bins) * width)
    ax.bar(centres, counts, width=np.deg2rad(width) * 0.95, color="k", lw=0)
    if period < 360:  # the first bin wraps around: draw its other half at `period`
        ax.bar(np.deg2rad(period), counts[0], width=np.deg2rad(width) * 0.95, color="k", lw=0)
    ax.set_thetamin(0)
    ax.set_thetamax(period)
    ax.set_xticks(np.deg2rad(np.arange(0, period + 1 if period < 360 else period, 45)))
    ax.set_rlabel_position(period / n_bins / 2)  # radial labels between the first two bars
    ax.yaxis.set_major_locator(plt.MaxNLocator(3))
    ax.tick_params(labelsize=5)
    return counts


def log_scatter(ax, x, y, stat, xlabel, ylabel):
    ax.scatter(x, y, s=8, color="k", alpha=0.6, linewidths=0)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ci = stat["ci"]
    ax.set_title(
        f"n = {stat['n']}, ρ = {stat['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}], bootstrap p = {stat['p']:.2g}",
        fontsize=6,
    )

In [ ]:
# summary figure statistics: cell selections, Venn counts, B and C correlations
from cottage_analysis.analysis import common_utils


def bootstrap_spearman(x, y, sessions):
    # Spearman rho, 95% CI and p from a hierarchical bootstrap over mice then sessions (thesis stats)
    df = pd.DataFrame(dict(x=np.asarray(x), y=np.asarray(y), session=np.asarray(sessions)))
    df = df[np.isfinite(df.x) & np.isfinite(df.y)].reset_index(drop=True)
    rho, dist = common_utils.hierarchical_bootstrap_stats(
        df, N_BOOTS, ["x"], ["mouse", "session"], ycol=["y"], correlation=True, plot=False
    )
    return dict(
        n=len(df),
        n_sessions=df.session.nunique(),
        rho=float(rho[0]),
        ci=np.percentile(dist[:, 0], [2.5, 97.5]),
        p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
    )


tuned = cells[cells.sftf_tuned & cells.depth_tuned]  # D, E
dir_tuned = tuned[tuned.fit_dsi > DSI_THR]  # F
both = cells[cells.sftf_tuned & cells.rsof_tuned].reset_index(drop=True)  # B, C

venn = dict(
    n_all=len(cells),
    n_depth=int(cells.depth_tuned.sum()),
    n_sftf=int(cells.sftf_tuned.sum()),
    n_both=int((cells.depth_tuned & cells.sftf_tuned).sum()),
)
venn["n_both_expected"] = venn["n_depth"] * venn["n_sftf"] / venn["n_all"]  # if independent
stat_speed_of = bootstrap_spearman(both.pref_speed, both.preferred_of, both.session)
stat_speed_depth = bootstrap_spearman(both.pref_speed, both.preferred_depth * 100, both.session)

print(venn)
for name, st in [("grating speed vs preferred OF", stat_speed_of), ("grating speed vs preferred depth", stat_speed_depth)]:
    print(f"{name}: n = {st['n']}, rho = {st['rho']:.3f}, 95% CI {st['ci'].round(3)}, p = {st['p']:.3g}")
print(f"D, E: {len(tuned)} cells; F: {len(dir_tuned)} cells")

In [ ]:
fig = plt.figure(figsize=(18 * CM, 12 * CM))
axes = {
    "A": fig.add_subplot(2, 3, 1),
    "B": fig.add_subplot(2, 3, 2),
    "C": fig.add_subplot(2, 3, 3),
    "D": fig.add_subplot(2, 3, 4),
    "E": fig.add_subplot(2, 3, 5, projection="polar"),
    "F": fig.add_subplot(2, 3, 6, projection="polar"),
}
plot_venn(
    axes["A"],
    venn["n_all"],
    venn["n_depth"],
    venn["n_sftf"],
    venn["n_both"],
    names=["Depth\nselective", "SF/TF\ntuned"],
    colors=["#2a78d6", "#1baf7a"],
)
log_scatter(axes["B"], both.pref_speed, both.preferred_of, stat_speed_of,
            "Preferred grating speed (deg/s)", "Preferred optic flow speed (deg/s)")
lims = [min(axes["B"].get_xlim()[0], axes["B"].get_ylim()[0]), max(axes["B"].get_xlim()[1], axes["B"].get_ylim()[1])]
axes["B"].plot(lims, lims, color="#8a8984", lw=0.8, ls="--", zorder=0)
log_scatter(axes["C"], both.pref_speed, both.preferred_depth * 100, stat_speed_depth,
            "Preferred grating speed (deg/s)", "Preferred virtual depth (cm)")

axes["D"].hist(tuned.fit_dsi, bins=20, range=(0, 1), color="k")
axes["D"].set_xlabel("Direction selectivity index")
axes["D"].set_ylabel("Number of neurons")
axes["D"].spines[["top", "right"]].set_visible(False)

polar_hist(axes["E"], tuned.fit_ori.values, 180, 4)
axes["E"].set_title("Preferred orientation (deg)", fontsize=7)
polar_hist(axes["F"], dir_tuned.fit_dir.values, 360, 8)
axes["F"].set_title("Preferred direction (deg)", fontsize=7)

for letter, ax in axes.items():
    ax.text(-0.15, 1.08, letter, transform=ax.transAxes, fontweight="bold", fontsize=9, va="bottom")
fig.tight_layout()
style.savefig(SAVE_ROOT / "fig_sftf_summary.pdf", fig=fig, bbox_inches="tight")

## Example neurons

Layout of the thesis example figure (`figure_utils/sftf_examples.py`): depth tuning and optic flow tuning
(closed-loop depth protocol), then mean grating responses (left) and the fitted model (right) for the 8
directions around a polar plot. One depth-selective and one non-depth-selective SF/TF-tuned cell, both
`iscell`. ROI 544 is the thesis example; the thesis depth-selective example (ROI 229 of the same session)
touches the edge of the registered field of view and is not `iscell`.

In [ ]:
from v1_depth_map.figure_utils import sftf_examples

# (session, roi): y-axis rounding of the depth tuning curve
EXAMPLES = {
    ("PZAH10.2f_S20230914", 447): dict(ylim_precision_base=0.1, ylim_precision=2),  # depth selective
    ("PZAH10.2f_S20230817", 544): dict(ylim_precision_base=0.1, ylim_precision=2),  # not depth selective
}
print(cells.set_index(["session", "roi"]).loc[list(EXAMPLES),
      ["depth_tuned", "preferred_depth", "sftf_tuned", "fit_sftf_test_rsq", "fit_dsi", "fit_dir"]].round(2))

example_data = {s: sftf_examples.load_example_session(flm, s) for s in {s for s, _ in EXAMPLES}}
fig = plt.figure(figsize=(18 * CM, 18 * CM))
for i, ((session, roi), ylim) in enumerate(EXAMPLES.items()):
    sftf_examples.plot_example_neuron(
        fig, example_data[session], roi, top=1 - i * 0.65,
        fontsize_dict={"title": 7, "label": 7, "tick": 5, "legend": 5}, **ylim,
    )
style.savefig(SAVE_ROOT / "fig_sftf_examples.pdf", fig=fig, bbox_inches="tight")

## Proportions of responsive and selective cells

In [ ]:
# proportions per group and χ² depth tuned vs not
SELECTIVITY = ["responsive", "ori_selective", "dir_selective"]
counts = cells.groupby("group")[SELECTIVITY].agg(["sum", "mean"])
counts.insert(0, "n", cells.groupby("group").size())
display(counts)

per_session = cells.groupby(["session", "group"])[SELECTIVITY].mean().unstack("group")
display(per_session.round(2))

prop_tests = {}
for col in SELECTIVITY:
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(cells.group, cells[col]))
    prop_tests[col] = dict(chi2=chi2, p=p)
    print(f"{col:14s} depth tuned vs not: chi2 = {chi2:.1f}, p = {p:.2g}")

In [ ]:
labels = ["responsive", "orientation\nselective", "direction\nselective"]
fig, ax = plt.subplots(figsize=(8 * CM, 5 * CM))
width = 0.38
for i, (group, color) in enumerate(GROUPS.items()):
    x = np.arange(len(SELECTIVITY)) + (i - 0.5) * width
    ax.bar(x, counts.loc[group, [(c, "mean") for c in SELECTIVITY]].values, width=width * 0.92,
           color=color, label=group)
    sess = per_session.xs(group, axis=1, level="group")[SELECTIVITY]
    for j, c in enumerate(SELECTIVITY):
        ax.scatter(np.full(len(sess), x[j]), sess[c], s=6, color="k", zorder=3, linewidths=0)
ax.set_xticks(np.arange(len(SELECTIVITY)), labels)
ax.set_ylabel("Proportion of cells")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
style.savefig(SAVE_ROOT / "proportions.pdf", fig=fig, bbox_inches="tight")

## Preferred orientation and direction

Orientation preference of orientation-selective cells and direction preference of direction-selective
cells, one bin per stimulus orientation / direction (the vector sum is continuous, but only 8 directions
were shown). Comparison: χ² on the binned counts.

In [ ]:
# binned preferences per group and χ² depth tuned vs not
def binned_counts(values, period, n_bins):
    # counts in bins centred on multiples of period / n_bins
    width = period / n_bins
    edges = np.arange(n_bins + 1) * width - width / 2
    return np.histogram((values + width / 2) % period - width / 2, bins=edges)[0], edges


PREFERENCES = {
    "orientation": dict(col="pref_ori", sel="ori_selective", period=180, n_bins=4),
    "direction": dict(col="pref_dir", sel="dir_selective", period=360, n_bins=8),
}
pref_counts, pref_tests = {}, {}
for name, d in PREFERENCES.items():
    pref_counts[name] = {
        group: binned_counts(cells.loc[(cells.group == group) & cells[d["sel"]], d["col"]].values,
                             d["period"], d["n_bins"])
        for group in GROUPS
    }
    table = np.array([c for c, _ in pref_counts[name].values()])
    chi2, p, _, _ = stats.chi2_contingency(table + 0.5)
    pref_tests[name] = dict(chi2=chi2, p=p, n=int(table.sum()))
    print(f"preferred {name}: {dict(zip(GROUPS, table.tolist()))}, chi2 p = {p:.2g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16 * CM, 5 * CM))
for ax, (name, d) in zip(axes, PREFERENCES.items()):
    for group, color in GROUPS.items():
        c, edges = pref_counts[name][group]
        ax.step(np.append(edges[:-1], edges[-1]), np.append(c, c[-1]) / c.sum(),
                where="post", color=color, lw=1.5, label=f"{group} (n={c.sum()})")
    ax.set_title(f"Preferred {name} ({d['sel'].replace('_', ' ')})\nχ² p = {pref_tests[name]['p']:.2g}",
                 fontsize=7)
    ax.set_xticks(np.arange(0, d["period"] + 1, 45))
    ax.set_ylim(0, None)
    ax.set_xlabel(f"Preferred {name} (deg)")
    ax.set_ylabel("Proportion of cells")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=6)
style.savefig(SAVE_ROOT / "preferred_orientation_direction.pdf", fig=fig, bbox_inches="tight")

## Selectivity indices

gOSI and gDSI of responsive cells (Mann-Whitney U).

In [ ]:
# gOSI / gDSI of responsive cells per group, Mann-Whitney U depth tuned vs not
index_samples, index_tests = {}, {}
for col in ["gosi", "gdsi"]:
    index_samples[col] = {
        group: np.sort(cells.loc[(cells.group == group) & cells.responsive, col].dropna().values)
        for group in GROUPS
    }
    a, b = index_samples[col].values()
    index_tests[col] = dict(p=stats.mannwhitneyu(a, b).pvalue, median=[np.median(a), np.median(b)])
    print(f"{col}: medians {np.round(index_tests[col]['median'], 3)}, p = {index_tests[col]['p']:.2g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12 * CM, 5 * CM))
for ax, col in zip(axes, ["gosi", "gdsi"]):
    for group, color in GROUPS.items():
        v = index_samples[col][group]
        ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=color, lw=1.5, label=f"{group} (n={len(v)})")
    ax.set_title(f"{col} (responsive cells), p = {index_tests[col]['p']:.2g}", fontsize=7)
    ax.set_xlabel(col)
    ax.set_ylabel("Cumulative proportion")
    ax.set_xlim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=6)
style.savefig(SAVE_ROOT / "selectivity_indices.pdf", fig=fig, bbox_inches="tight")

## Preferred orientation vs preferred depth

Depth-tuned, orientation-selective cells. Independent preferences would show as no structure here.

In [ ]:
# preferred depth across the 4 orientation bins (centred on the stimulus orientations), Kruskal-Wallis
ori_depth = cells[cells.depth_tuned & cells.ori_selective]
ori_bin = ((ori_depth.pref_ori + 22.5) % 180 // 45 * 45).astype(int)
kw = stats.kruskal(*[np.log(ori_depth.preferred_depth[ori_bin == b]) for b in sorted(ori_bin.unique())])
ori_depth_test = dict(n=len(ori_depth), H=kw.statistic, p=kw.pvalue)
print(ori_depth_test)

In [ ]:
fig, ax = plt.subplots(figsize=(6 * CM, 5 * CM))
ax.scatter(ori_depth.preferred_depth, ori_depth.pref_ori, s=6, color=GROUPS["depth tuned"], linewidths=0)
ax.set_xscale("log")
ax.set_xlabel("Preferred virtual depth (m)")
ax.set_ylabel("Preferred orientation (deg)")
ax.set_yticks(np.arange(0, 181, 45))
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"n = {ori_depth_test['n']}, Kruskal-Wallis across orientation bins p = {ori_depth_test['p']:.2g}",
             fontsize=7)
style.savefig(SAVE_ROOT / "orientation_vs_depth.pdf", fig=fig, bbox_inches="tight")

## Export statistics

Every value is read from a variable computed above for a panel; nothing is recomputed. Written to
`v1_depth_map/stats/generated/stats_fig_reviewer_gratings.yaml` (group `reviewer`, after the manuscript
figures); `python -m v1_depth_map.stats.build_manuscript_stats` compiles it into the dashboard and the
LaTeX macros.

In [ ]:
from v1_depth_map.stats import export_figure_stats


def _fmt_pval(p):
    # p-value as it should read in LaTeX, in math mode so it works in text or math
    p = float(p)
    return "$p < 0.0001$" if p < 1e-4 else f"$p = {p:.3g}$"


def _stat(raw, formatted, macro, description):
    return dict(raw=raw, formatted=formatted, latex_macro=macro, description=description)


def _count(n, macro, description):
    return _stat(int(n), f"{int(n):,}", macro, description)


def _pct(fraction, macro, description):
    return _stat(float(100 * fraction), f"{100 * fraction:.0f}\\%", macro, description)


def _p(p, macro, description):
    return _stat(float(p), _fmt_pval(p), macro, description)


def _rho(r, macro, description):
    return _stat(float(r), f"{r:.2f}", macro, description)


n_all = venn["n_all"]
groups_n = cells.group.value_counts()
stats_reviewer = {
    "n_neurons": _count(n_all, "statRevGratNeurons", "iscell neurons in the 6 grating sessions"),
    "n_sessions": _count(cells.session.nunique(), "statRevGratSessions", "Sessions with drifting gratings"),
    "n_mice": _count(cells.session.str.split("_").str[0].nunique(), "statRevGratMice", "Mice with drifting gratings"),
    "n_depth": _count(venn["n_depth"], "statRevGratDepthNeurons", "Depth-selective neurons (A)"),
    "pct_depth": _pct(venn["n_depth"] / n_all, "statRevGratPctDepth", "Percentage depth selective (A)"),
    "n_sftf": _count(venn["n_sftf"], "statRevGratSftfNeurons", "SF/TF-tuned neurons, cross-validated (A)"),
    "pct_sftf": _pct(venn["n_sftf"] / n_all, "statRevGratPctSftf", "Percentage SF/TF tuned (A)"),
    "pct_both": _pct(venn["n_both"] / n_all, "statRevGratPctBoth", "Percentage depth selective and SF/TF tuned (A)"),
    "pct_both_expected": _pct(venn["n_both_expected"] / n_all, "statRevGratPctBothExpected",
                              "Percentage both expected if independent (A)"),
    "speed_of_n": _count(stat_speed_of["n"], "statRevGratSpeedOfN", "SF/TF- and RS/OF-tuned neurons (B)"),
    "speed_of_rho": _rho(stat_speed_of["rho"], "statRevGratSpeedOfRho",
                         "Spearman rho, preferred grating speed vs preferred optic flow (B)"),
    "speed_of_p": _p(stat_speed_of["p"], "statRevGratSpeedOfP", "Hierarchical bootstrap p (B)"),
    "speed_depth_rho": _rho(stat_speed_depth["rho"], "statRevGratSpeedDepthRho",
                            "Spearman rho, preferred grating speed vs preferred depth (C)"),
    "speed_depth_p": _p(stat_speed_depth["p"], "statRevGratSpeedDepthP", "Hierarchical bootstrap p (C)"),
    "n_tuned_depth": _count(len(tuned), "statRevGratTunedDepthN", "SF/TF-tuned depth-selective neurons (D, E)"),
    "n_dir_tuned": _count(len(dir_tuned), "statRevGratDirTunedN", f"Of which dsi > {DSI_THR} (F)"),
}
for col, macro in [("responsive", "Resp"), ("ori_selective", "OriSel"), ("dir_selective", "DirSel")]:
    stats_reviewer[f"{col}_pct_depth"] = _pct(
        counts.loc["depth tuned", (col, "mean")], f"statRevGrat{macro}PctDepth", f"Percentage {col}, depth tuned")
    stats_reviewer[f"{col}_pct_other"] = _pct(
        counts.loc["not depth tuned", (col, "mean")], f"statRevGrat{macro}PctOther", f"Percentage {col}, not depth tuned")
    stats_reviewer[f"{col}_p"] = _p(prop_tests[col]["p"], f"statRevGrat{macro}P", f"chi2 p, {col} depth tuned vs not")
stats_reviewer["pref_ori_p"] = _p(pref_tests["orientation"]["p"], "statRevGratPrefOriP",
                                  "chi2 p, preferred orientation depth tuned vs not")
stats_reviewer["pref_dir_p"] = _p(pref_tests["direction"]["p"], "statRevGratPrefDirP",
                                  "chi2 p, preferred direction depth tuned vs not")
stats_reviewer["gosi_p"] = _p(index_tests["gosi"]["p"], "statRevGratGosiP", "Mann-Whitney p, gOSI depth tuned vs not")
stats_reviewer["gdsi_p"] = _p(index_tests["gdsi"]["p"], "statRevGratGdsiP", "Mann-Whitney p, gDSI depth tuned vs not")
stats_reviewer["ori_depth_n"] = _count(ori_depth_test["n"], "statRevGratOriDepthN",
                                       "Depth-tuned orientation-selective neurons")
stats_reviewer["ori_depth_p"] = _p(ori_depth_test["p"], "statRevGratOriDepthP",
                                   "Kruskal-Wallis p, preferred depth across orientation bins")

export_figure_stats("fig_reviewer_gratings", stats_reviewer, figure="reviewer")